# Open fragmentation, conservative engine, geometric kernel

Split from `Run_open_no_resampling_Kernal_Geometric.ipynb`.  The matching analysis notebook is
`Analysis_open_fragmentation_conservative_geometric.ipynb`.


# Majorant\_v2 — open cascades on the conservative engine

This notebook runs the two open configurations, coagulation and fragmentation, on
`BF_v_no_resampling_v2.py`. One simulated particle stands for exactly one physical
particle for the whole run, so the total number and the total mass are conserved to
machine precision and the reported `mass_drift` is identically zero; anything else is
a bug rather than a fluctuation. That is what *conservative* means in the names of
the files this notebook writes.

An open system has a source and an absorbing boundary, so a genuine steady state
exists and the spectrum to measure is the **instantaneous** one rather than a
superposition over time. $\Delta t$ never enters the estimator, which is why the
closed notebook has to agonise over the snapshot cadence and this one does not — and
which is also why every snapshot taken after the cascade reached the sink is an
independent draw from the same distribution, so they are averaged rather than thrown
away. That average costs nothing and is worth a factor $\sqrt{K}$ in noise. The theory
under test is

$$\alpha = -\frac{3+\lambda}{2}, \qquad b = \frac{2}{1-\lambda},$$

with $\lambda$ the homogeneity degree of the kernel, $\alpha$ the slope of $dN/dm$
across the inertial range and $b$ the growth exponent read from the isochrones.
Coagulation injects monomers at $m_{\rm inj}$ and absorbs products above
$m_{\rm sink}$; fragmentation injects large bodies at $m_{\rm inj}$ and absorbs
fragments below $m_{\rm sink}$. The two share $\alpha$ and $b$ exactly, and only the
direction of the drift changes.

Spectra are drawn compensated, as $m^2\,dN/dm$ — the mass per logarithmic mass
interval, whose slope is $\alpha+2$. Every fit is performed on the raw $dN/dm$ and
the compensation is applied at draw time only, so the reported index is never the
compensated one.

Each successful run is written into `runs/` by `add_last_run`, keeping the spectra,
the isochrone histogram and all the parameters but discarding the per-particle
arrays. The analysis notebooks in that folder rebuild every figure from those files,
so a plot can be changed without paying for the simulation again.


## How the index is measured

A measured spectrum is a power law only *between* the two characteristic masses of
the problem. Outside that band it bends — at the injection scale because of the
source, at the sink because of truncation and vanishing statistics — and a single fit
across the whole array averages the plateau together with both bends, returning a
number that describes neither. What is measured here instead is the plateau in the
local slope,

$$\Gamma(m)=\frac{d\log F}{d\log m},$$

computed by least squares on a sliding window. `BF.find_inertial_range` returns the
longest contiguous stretch over which $\Gamma$ stays flat within a tolerance,
together with that stretch's mean, its scatter and its width in decades. The mean is
the measured $\alpha$: it is what enters the summary table and it is the line drawn
over the spectrum. When it comes back as `nan` the run simply has no inertial range
yet, which is the correct answer rather than a failure. What it is computed on is
`steady_spectrum`, the mean of the snapshots taken after the sink gate opened, not a
single snapshot.

Running alongside it is the a-priori guard band, half a decade stripped from each end
of the interval between $m_{\rm inj}$ and $m_{\rm sink}$, chosen before anyone looks
at the data. The two must agree; where they do not, the cascade has not developed
over the range that was assumed. The width in decades is printed next to every index
because a plateau narrower than roughly one decade is not a power law however tight
its error bar looks, and because the plateau finder can only test whether the curve
you computed has a scaling region — not whether that curve is the right quantity to
have computed.


In [ ]:
import os, importlib, inspect, numpy as np, matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# ============================================================================
#  КАКОЙ ДВИЖОК.  Одна строка, всё остальное подстраивается ниже.
# ============================================================================
#   "warm"  BF_warm_start_v5.py -- популяция ЗАСЕВАЕТСЯ степенным законом, а не
#           строится из инжекции.  Даёт три вещи:
#             * alpha и закон ожидания <dt>(m) честны почти сразу, а не через
#               одно время пребывания tau_res (на шести декадах это 45 оборотов);
#             * метка честности gen = -1 наследуется, так что посеянные тела и
#               все их потомки НЕ попадают в статистику поколений -- поколения
#               по-прежнему требуют промывки, и honest_num говорит, насколько
#               она прошла;
#             * <dt>(m) ~ m^(1/b): b как ВРЕМЯ, а не через alpha.
#   "cold"  BF_v_no_resampling_v4.py -- как было: дельта при m_inj, ждём.
#
#  ЧТО ТЁПЛЫЙ СТАРТ НЕ ПОКУПАЕТ.  Промывка коробки стоит M_sys/m_sink событий
#  всегда.  Тёплый старт снимает её с alpha и с <dt>, но НЕ с поколений: чтобы
#  gen был честным, посеянное должно уйти.  Это не оговорка, а половина плана
#  прогона.
ENGINE = "warm"        # <<-- "warm" | "cold"

# ============================================================================
#  С ЧЕГО НАЧИНАТЬ.  Три варианта, и они не равноценны.
# ============================================================================
#   "cold"    дельта при m_inj.  У каждой частицы счётчик разломов честный с
#             первого события, потому что все вошли одинаково.  Цена -- надо
#             дождаться стационара, а это НЕ одна промывка коробки (см. ниже).
#   "seeded"  синтетический степенной посев.  Спектр правильный сразу, но
#             истории нет ни у кого: gen = -1 наследуется, поколения слепы,
#             пока посев не уйдёт.  Годится для alpha и <dt>(m), не для gen.
#   "state"   ПЕРЕЗАПУСК с сохранённого стационарного состояния.  Всё лучшее от
#             обоих: спектр правильный с t = 0 И у всех честная история, потому
#             что состояние получено холодным прогоном.  Возрасты переносятся
#             через шов отрицательными gen_time, поэтому tau(g) НЕ обрезается
#             длиной прогона -- именно это дало b = 1.74 вместо 6 в v4.
#
#  РАБОЧИЙ ПОРЯДОК.  Не угадывать длину заранее, а идти прогонами:
#     START="cold", SAVE_STATE=True  -> прогон -> AN.stationarity() -> не сошлось?
#     START="state", SAVE_STATE=True -> ещё прогон -> проверить -> ... -> сошлось
#     START="state", SAVE_STATE=False -> измерительный прогон
#  Перезапуски составляются: t_origin копит время всех предыдущих прогонов.
#
#  СКОЛЬКО ЖДАТЬ СТАЦИОНАРА.  Замерено на холодном шестидекадном прогоне,
#  в промывках коробки (одна промывка = M_sys/m_sink событий):
#        промывок   <m>      alpha    dn_out/dev   M_out/M_in
#          0.18     16.2    -1.890       0.96
#          0.46     22.5    -1.881       1.01
#          1.01     26.6    -1.862       1.02
#          1.29     30.6    -1.829       1.02         1.016
#  Оба привычных критерия отрапортовали успех на 0.2 промывки, а <m> при этом
#  росло по 14 за промывку и alpha ехало, в конце быстрее, чем в начале.
#  M_out/M_in = 1 + dM_sys/M_in стремится к единице просто потому, что M_in
#  растёт, а dn_out/dev -- это баланс ЧИСЛА, которое живёт у стока и садится
#  за t_turn, тогда как форма живёт наверху и садится за tau_res, в <m>/m_sink
#  раз дольше.  Судить только по <m> и alpha: AN.stationarity(r4).
START      = "state"    # <<-- "cold" | "seeded" | "state"
SAVE_STATE = True       # <<-- сохранить состояние частиц в конце (файл ~100+ МБ)
STATE_FILE = "runs/state_open_fragmentation_conservative_geometric_f0.30.npz"

if ENGINE == "warm":
    import BF_warm_start_v5 as BF
elif ENGINE == "cold":
    import BF_v_no_resampling_v4 as BF
else:
    raise ValueError("ENGINE = 'warm' или 'cold'")
import BF_analysis as AN
importlib.reload(BF); importlib.reload(AN)   # кэш модуля -- молча другой ответ, не ошибка

plt.rcParams.update({"figure.dpi":110, "font.size":9, "axes.grid":True,
                     "grid.alpha":0.25, "figure.figsize":(9,3.2)})

KERNEL = BF.kernel_geometric         # lambda = 2/3 (geometric cross section)
LAM    = BF.KERNEL_LAMBDA[KERNEL.__name__]

# GRID.  Trimmed from 10**arange(-4,12,0.1): that grid carried 59 bins above m_inj and
# 40 below the sink which stayed empty for the whole run, while every O(B) vector
# operation and the whole fmaj matrix paid for them.  Clipping from BELOW is safe --
# the corner majorant only grows -- and nothing exceeds m_inj in fragmentation.
EDGES  = 10.0 ** np.arange(-1, 6.45, 0.1)

_par = inspect.signature(BF.simulate).parameters
assert "track_generations" in _par, "движок на пути не несёт поколений (нужен v4 или v5)"
WARM = "track_waiting" in _par
assert (ENGINE == "warm") == WARM, \
    "ENGINE = %r, а на пути %s -- проверь, что BF_warm_start_v5.py лежит рядом" % (ENGINE, BF.__name__)

if START not in ("cold", "seeded", "state"):
    raise ValueError("START = 'cold' | 'seeded' | 'state'")
if START in ("seeded", "state") and not WARM:
    raise ValueError("START=%r требует ENGINE='warm' (BF_warm_start_v5)" % START)
if START == "state" and not hasattr(BF, "state_of"):
    raise ValueError("движок на пути не умеет перезапуск с состояния")

print("engine =", BF.__name__, "|", len(_par), "параметров |", "старт:", START)
print("kernel =", KERNEL.__name__, " lambda =", LAM)
PR = BF.predict("open", LAM)
print("  open   beta = %.4g   b = %.4g   alpha = %.4g" % (PR["beta"], PR["b"], PR["alpha"]))
print("  grid   %.3g .. %.3g  (%d bins of %.2f dex)"
      % (EDGES[0], EDGES[-1], EDGES.size-1, np.log10(EDGES[1]/EDGES[0])))

#  ТРИ ДОРОГИ К b, и все три печатаются в конце.  Полезно держать их рядом с самого
#  начала, потому что они меряют РАЗНОЕ и расходятся там, где это что-то значит.
#     замыкание   1/b = 2 + alpha           -- поток, через плотность
#     кинетика    1/b = -(1 + lambda + alpha) -- скорость события при том же alpha
#     измерение   1/b = dln<dt>/dln m        -- время, напрямую   [только v5]
#  На неподвижной точке alpha = -(3+lambda)/2 первые две совпадают и дают
#  1/b = (1-lambda)/2.  Вне её -- нет, и разрыв между ними и есть мера того,
#  насколько прогон ещё не дошёл.
print("  b      теория 1/b = (1-lambda)/2 = %.5f  ->  b = %.3f" % (0.5*(1-LAM), 2/(1-LAM)))
print("         db/dalpha = b^2 = %.0f: один процент на alpha -- целая единица на b."
      % (2/(1-LAM))**2)
print("         поэтому везде ниже цитируется 1/b с ошибкой, а b -- производное число.")

# Age bins for the isochrones.  One age bin of width D_tau dex covers b*D_tau dex of
# MASS, so at b = 6 a 0.15 dex step jumps 0.9 dex of mass per bin.  Fix the MASS
# resolution and let the age step follow.  The GENERATION histogram needs none of this:
# its axis is an integer and it is re-binnable in the analysis for free.
AGE_STEP = 0.30 / PR["b"]
print("  ages   %.3f dex per bin  ->  %.2f dex of mass per bin" % (AGE_STEP, 0.30))

RESULTS = {}


In [ ]:
# ----------------------------------------------------------------------
#  The measurement layer
# ----------------------------------------------------------------------
#  Everything that turns arrays into a number now lives in BF_analysis.py, next to
#  this notebook: the estimator and its error bars, the plateau finders, the
#  isochrone machinery and the saver.  Keeping it there rather than in a cell means
#  the run notebooks and the six analysis notebooks cannot drift apart, and that a
#  change to the estimator is one edit rather than nine.
#
#  The names are aliased below so the figure cells further down read exactly as
#  before.

import BF_analysis as AN

anchor_amplitude = AN.anchor_amplitude
pick_isochrones  = AN.pick_isochrones
draw_isochrones  = AN.draw_isochrones
compensated_ylim = AN.compensated_ylim
add_last_run     = AN.add_last_run


def steady_spectrum(run, frac=0.5):
    """
    (F, K): the averaged post-gate spectrum and how many snapshots went into it.

    AN.steady_spectrum returns the full dict -- F, the empirical and Poisson error
    bars, the raw counts and K_eff.  This shim keeps the two-value call used below;
    call AN.steady_spectrum directly when the error bars are wanted.
    """
    s = AN.steady_spectrum(run, frac=frac)
    return s["F"], s["K"]


def check_grid(*masses):
    """Refuse to start if a characteristic mass falls outside EDGES."""
    return AN.check_grid(EDGES, *masses)


---
## Open system with fragmentation

Large bodies are injected at $m_{\rm inj}$ and fragments falling below
$m_{\rm sink}$ are absorbed — the mirror of the coagulation case, predicting the same
$\alpha$. This one is run twice, because the rule deciding which particle breaks is a
choice of physics rather than a detail of implementation.

If the heavier of the pair always breaks, whatever hit it, then the disruption rate
of a body of mass $m_0$ against a background $F(m)\propto m^{\alpha}$ is

$$\nu(m_0)\sim\int_{m_{\rm sink}}^{m_0}F(m')\,K(m_0,m')\,dm'
        \sim\int_{m_{\rm sink}/m_0} x^{\alpha}\,dx ,$$

which diverges at the lower limit whenever $\alpha<-1$. The drift is then set by the
smallest particles in the box — by the sink scale rather than by $m_0$ — the
mean-field closure is void, and the measured index locks onto $-2$ for any kernel at
all. That is a boundary effect wearing the costume of universality.

The cure is the one Dohnanyi built in: a catastrophic-disruption threshold, on the
grounds that a grain of dust does not shatter a boulder. Requiring
$m_{\rm small}\ge f\,m_{\rm large}$ with $f$ of order a tenth to one restricts the
integral to $x\in[f,1]$, which is scale free, and recovers the predicted index. A
closed system is immune by construction, because its packet is narrow and every
partner is already of order $m_0$ — which is why the closed runs agree with the
theory at $f=0$ and the open ones do not.
The cell below runs the local rule only, $f=0.3$. The $f=0$ branch has been dropped:
without a threshold there is no steady cascade to measure at all — the last such run
ground its whole population into $1<m<447$ and returned $\alpha=-4.7$ over nine tenths
of a decade, which is the shape of a pile at the sink and not an index.

### Where the run starts

$N_0$ is not a detail. The clock is $dt = 2V/(wR)$ with $R\sim N^2$, so the physical
time consumed while the box fills goes as $1/N_0$, and the deterministic injector
delivers $q\,t$ bodies during it. Measured on this engine at three decades:

| $N_0$ | bodies injected during the fill |
|---|---|
| 4 | 705 |
| 30 | 329 |
| 180 | 200 |
| 1000 | 38 |

The previous version of this cell used $N_0=4$, and three quarters of every gram that
ever entered the box entered in the first thousand trials; after that the injector was
effectively off — nine bodies per eighteen million events — the workload was set by that
burst rather than by the sink budget asked for, and the run was still filling when it was
stopped: `live` at $1.07\times10^7$ against a target of $4\times10^6$ and rising, with
$dn_{\rm out}/d\,\text{events}=0.43$ where a steady state needs 1.

$N_0$ is now chosen from the mass the box has to hold rather than guessed.
`BF.steady_mean_mass` gives $\langle m\rangle = 45$ at $\alpha=-11/6$ over six decades,
so four million particles want 180 bodies of $10^6$. Whether it worked is checked after
the run rather than assumed.

In [ ]:
# ----------------------------------------------------------------------------
#  Шесть декад: инжекция при m_inj = 1e6, поглощение при m_sink = 1.
# ----------------------------------------------------------------------------
#  live = N0 + n_injected + events - n_out, до последней единицы.  В стационаре
#  dn_out/devents = 1: событие делает частицу, сток обязан её забрать.  Сильно ниже
#  единицы при растущем live -- популяция копится, и прогон переходный сколько его ни
#  жди.  Проверяется в конце ячейки, а не на глаз.
#
#  N0 -- НЕ деталь.  dt = 2V/(wR), R ~ N^2, поэтому время заполнения идёт как 1/N0, а
#  инжектор за это время выдаёт q*t тел.  Замерено на этом движке, три декады:
#        N0 =    4  ->  705 тел за заполнение
#        N0 =  180  ->  200
#        N0 = 1000  ->   38
#  Поэтому N0 берётся из массы, которую коробка обязана держать.
import time

M_INJ4, M_SINK4 = 1.0e6, 1.0
R4       = M_INJ4 / M_SINK4
N_SS4    = 4.0e6                 # целевая живая популяция; она же задаёт память
W_SPLIT  = 0.5                   # полуширина дробления; ниже 0.25 alpha сдвигается
F_RATIO4 = 0.3                   # порог разрушения; f = 0 нелокален
N_OUT4   = 5.0e8                 # поглощений на стоке -- тормоз ИЗМЕРИТЕЛЬНОГО прогона
FLUSHES4 = 2.0                   # длина СТРОЯЩЕГО прогона, в промывках коробки
                                 # (одна промывка = M_sys/m_sink событий).  Это
                                 # та же единица, в которой отчитывается
                                 # AN.stationarity, поэтому «ещё 0.7 промывки»
                                 # переводится в FLUSHES4 без пересчёта.
HOURS4   = 9.0
SEED4    = 4
FRAG_AGE_RULE = "heavier"        # только ярлык для изохрон; alpha сдвинуть не может
GEN_MAX  = 192                    # выше -- в gen_overflow, не в последний бин

# ============================================================================
#  ЗАСЕВ  (работает только при ENGINE = "warm")
# ============================================================================
#  ЧЕСТНОСТЬ.  Засеять alpha = -11/6 и потом отрапортовать alpha = -1.83 --
#  значит не доказать ничего: стационарное состояние открытого каскада есть
#  АТТРАКТОР, и единственный способ это показать, а не предположить, -- посеять
#  заведомо НЕВЕРНЫЙ наклон и посмотреть, уйдёт ли он.  Поэтому по умолчанию
#  здесь стоит -1.50, а не -1.8333.  Прогоните оба края (-1.50 и -2.10) и
#  убедитесь, что оба приходят в одно место; тогда и только тогда засев -- не
#  подсказка ответа.
#
#  Что помечено.  Посеянные тела несут gen = -1, и -1 НАСЛЕДУЕТСЯ: осколок
#  посеянного тела и продукт слияния с ним тоже -1.  Заражение только
#  распространяется, никогда не лечится, поэтому honest_num -- честная нижняя
#  оценка того, какая доля популяции пригодна для поколений.
ALPHA_SEED = -1.50               # <<-- посеять НЕ ответ: -1.50 и -2.10, оба к -1.83
SEED_LO, SEED_HI = M_SINK4, M_INJ4

MBAR4  = AN.steady_mean_mass(PR["alpha"], M_SINK4, M_INJ4)
M_SYS4 = N_SS4 * MBAR4
check_grid(M_INJ4, M_SINK4)

if START == "state":
    #  ПЕРЕЗАПУСК.  Состояние получено холодным прогоном, поэтому у каждой
    #  частицы счётчик разломов настоящий -- ничего не обнуляется.  Времена
    #  сдвигаются на t_end предыдущего прогона и уходят в минус: частицы вправду
    #  вошли до нуля, и tau = t - t_anc остаётся ПОЛНЫМ возрастом через шов.
    _prev = AN.load(STATE_FILE)
    STATE4 = BF.state_of(_prev, source=STATE_FILE)
    IC4 = {"state": STATE4}
    N04 = int(STATE4["mass"].size)
    _mb = float(STATE4["mass"].mean())
    _sp = AN.spectrum(_prev); _pl = AN.spectrum_plateau(_prev, spec=_sp)
    print("ПЕРЕЗАПУСК с %s" % STATE_FILE)
    print("  частиц %.4g | <m> = %.4g | масса %.4g | t уже накоплено %.4g"
          % (N04, _mb, STATE4["mass"].sum(),
             STATE4["t_origin"] + STATE4["t_end"]))
    print("  спектр состояния: alpha = %+.4f +- %.4f на %.2f декадах"
          % (_pl["alpha"], _pl["scatter"], _pl["decades"]))
    print("  поколения в состоянии: max g = %d, среднее %.1f"
          % (STATE4["gen"].max(), STATE4["gen"].mean()))
    _ss = AN.stationarity(_prev)
    print(_ss["report"])
    if not _ss["ok"]:
        print("  !! состояние НЕ стационарно.  Мерить с него можно, но это будет")
        print("     измерение переходного режима.  Лучше добить ещё один прогон.")
    if STATE4["gen"].max() > GEN_MAX:
        raise ValueError("в состоянии есть g = %d > GEN_MAX = %d -- подними GEN_MAX"
                         % (STATE4["gen"].max(), GEN_MAX))
    del _prev
elif START == "seeded":
    # Число тел берётся из МАССЫ, которую коробка обязана держать, а не наоборот:
    # <m> посеянного спектра зависит от ALPHA_SEED, и фиксировать N значило бы
    # менять массу коробки вместе с наклоном -- то есть менять два параметра там,
    # где хотели поменять один.
    N04, MBAR_SEED = BF.steady_N_for_mass(ALPHA_SEED, SEED_LO, SEED_HI, M_SYS4)
    IC4 = {"steady": {"alpha": ALPHA_SEED, "m_lo": SEED_LO, "m_hi": SEED_HI, "N": N04}}
    print("ЗАСЕВ: alpha_seed = %+.4f на [%.3g, %.3g], <m> = %.4g  ->  N0 = %.3g тел"
          % (ALPHA_SEED, SEED_LO, SEED_HI, MBAR_SEED, N04))
    print("       (стационарное <m> = %.4g при alpha = %.4f -- отличие и есть то,"
          % (MBAR4, PR["alpha"]))
    print("        что засев обязан пройти сам, а не получить в подарок)")
else:
    N04 = int(np.ceil(M_SYS4 / M_INJ4))
    IC4 = {"m": M_INJ4, "N": N04}
    print("ХОЛОДНЫЙ СТАРТ: N0 = %d тел при m_inj на %.3g массы" % (N04, M_SYS4))

q4      = 0.5 * KERNEL(M_SINK4, M_SINK4) * N_SS4**2 / R4
t_turn4 = N_SS4 / (q4 * R4)
tau_res4 = M_SYS4 / (q4 * M_INJ4)
t_c4    = t_turn4
AGE_EDGES4 = 10.0**np.arange(-5, 4, AGE_STEP) * t_turn4

print("population: N_ss = %.3g, <m> = %.3g  ->  масса коробки %.3g"
      % (N_SS4, MBAR4, M_SYS4))
#  ДВА ВРЕМЕНИ, и путать их -- ошибка на порядок.  t_turn = N_ss/(qR) заменяет ЧИСЛО
#  частиц, а число живёт у стока.  tau_res = M_sys/(q m_inj) заменяет МАССУ, а масса
#  живёт наверху.  Отличаются ровно в <m>/m_sink раз.
print("            q4 = %.4g | t_turn (число) = %.4g | tau_res (масса) = %.4g = %.0fx"
      % (q4, t_turn4, tau_res4, tau_res4 / t_turn4))
print("            память ~ %.0f МБ массивов + ~%.0f МБ списков бинов + %.0f МБ на gen"
      % (N_SS4 * 32 * 1.6 / 1e6, N_SS4 * 36 / 1e6, N_SS4 * 12 * 1.6 / 1e6))

#  ПОКОЛЕНИЯ ВМЕСТО ТРЕЙСЕРОВ.  Номер разлома несёт КАЖДАЯ частица, один int32.  Ни
#  деревьев, ни полки, ни цензурирования: частица считается пока жива, и её поколение
#  не зависит ни от того, сколько она ждала, ни от того, доживёт ли она до стока.
#  Сколько поколений будет пригодно -- считается заранее: пакет по МАССЕ едет со
#  скоростью <ln xi> = -1/2 за разлом, значит среднее дойдёт до стока на
GSINK = np.log(M_INJ4 / M_SINK4) / abs(AN.step_stats(W_SPLIT, "mass")[0])
print("generations: пакет по массе достигнет стока на g ~ %.0f, масштаб tau(g) = 2b = %.0f"
      % (GSINK, 2 * PR["b"]))
print("             для b нужно, чтобы пригодных поколений было заметно больше масштаба")

#  ВОРОТА.  На холодном старте они ждут, пока каскад построится.  На тёплом строить
#  нечего, поэтому ворота опускаются -- но только для СПЕКТРА и для <dt>.  Поколениям
#  всё равно нужна промывка, и следит за ней honest_num, а не ворота.
#  На ХОЛОДНОМ старте и на ПЕРЕЗАПУСКЕ история честна у всех с первого события,
#  поэтому ворота не нужны вовсе.  Они нужны только синтетическому посеву, и то
#  лишь чтобы отрезать самый грубый переходный кусок.
ISO_SINK4 = int(0.2 * N_SS4) if START == "seeded" else 0

_t0  = time.time()
_cal = BF.simulate(process="fragmentation", system="open", kernel=KERNEL, edges=EDGES,
                   ic=IC4, frag_min_ratio=F_RATIO4,
                   frag_split_width=W_SPLIT, injection_rate=q4, injection_mass=M_INJ4,
                   sink_mass=M_SINK4, snapshot_mode="events", snapshot_stride=1e12,
                   max_events=200_000, track_generations=False,
                   rng=np.random.default_rng(0), verbose=False)
RATE4 = float(_cal["events"][-1]) / max(time.time() - _t0, 1e-9)
print("probe     : %.0f событий при %.3g ev/s, acceptance %.3f"
      % (_cal["events"][-1], RATE4, _cal["acceptance"]))
del _cal

#  СТОИМОСТЬ.  Промывка коробки -- каждая единица массы уходит как m_sink частиц, по
#  одной на событие -- плюс собственно набор.  Оценка по потоку без промывки занижает:
#  на прошлом прогоне бюджет был 6.6e7, а реальная цена 2.2e8, и прогон уткнулся в
#  часы вместо стока.  Поэтому промывка ВХОДИТ в бюджет.
_flush = M_SYS4 / M_SINK4
CEIL4  = int(RATE4 * 3600.0 * HOURS4)
#  На ПЕРЕЗАПУСКЕ промывка уже оплачена предыдущими прогонами -- в бюджет входит
#  только набор статистики.  На холодном и на посеве она входит целиком.
#  ДВА РЕЖИМА, и различает их SAVE_STATE -- отдельного переключателя не надо.
#    SAVE_STATE=True  -- прогон СТРОИТ стационар.  Длина задаётся в промывках,
#                        потому что именно в них считается сходимость.
#    SAVE_STATE=False -- прогон МЕРИТ.  Длина задаётся набором на стоке; промывка
#                        входит в неё только если стартуем не с состояния.
if SAVE_STATE:
    _need = FLUSHES4 * _flush
else:
    _need = 1.2 * (N_OUT4 if START == "state" else _flush + N_OUT4)
MAX_EV4 = int(min(CEIL4, _need))
print("cost      : промывка %.3g событий + %.3g на сток = %.3g, это %.2f ч"
      % (_flush, N_OUT4, _flush + N_OUT4, (_flush + N_OUT4) / RATE4 / 3600.0))
print("            бюджет прогона %.3g событий = %.2f промывки = %.2f ч, упрётся в %s"
      % (MAX_EV4, MAX_EV4 / _flush, MAX_EV4 / RATE4 / 3600.0,
         ("промывки" if SAVE_STATE else "сток") if _need <= CEIL4
         else "ЧАСЫ -- подними HOURS4"))
if START == "state":
    print("            промывка (%.3g) в бюджет НЕ входит: она оплачена прогонами до этого"
          % _flush)
if START == "seeded":
    print("            засев: alpha и <dt> честны почти сразу, а поколениям всё")
    print("            равно нужна промывка -- следить за honest_num, не за воротами.")
    print("            доля промывки в бюджете: %.0f%%" % (100 * _flush / (_flush + N_OUT4)))
elif START == "state":
    print("            перезапуск: честна вся популяция с первого события, и")
    print("            возрасты пришли через шов -- tau(g) не обрезана длиной прогона.")

_t = time.time()
_kw = dict(track_waiting=True) if WARM else {}
r4 = BF.simulate(
    process="fragmentation", system="open", kernel=KERNEL, edges=EDGES,
    ic=IC4, frag_min_ratio=F_RATIO4, frag_split_width=W_SPLIT,
    frag_age_rule=FRAG_AGE_RULE, track_generations=True, gen_max=GEN_MAX,
    injection_rate=q4, injection_mass=M_INJ4, sink_mass=M_SINK4,
    snapshot_mode="events", snapshot_stride=max(int(MAX_EV4) // 200, 1),
    max_events=MAX_EV4, stop_sink_events=N_OUT4,
    iso_start_sink=ISO_SINK4, iso_age_edges=AGE_EDGES4,
    rng=np.random.default_rng(SEED4), verbose=True, **_kw)

print("\n%.1f мин | live %d->%d | sink=%d | M_out/M_in=%.3f | stop=%s"
      % ((time.time() - _t) / 60, r4["live"][0], r4["live"][-1], r4["sink_events"],
         r4["M_out"][-1] / max(r4["M_in"][-1], 1), r4["meta"]["stop_reason"]))

_ev, _lv, _no = r4["events"], r4["live"], r4["n_out"]
_h = _ev.size // 2
_slope = (_no[-1] - _no[_h]) / max(_ev[-1] - _ev[_h], 1)
_drift = (_lv[-1] - _lv[_h]) / max(_lv[_h], 1)
print("throughput: dn_out/dev = %.3f | дрейф live за 2-ю половину = %+.1f%%"
      % (_slope, 100 * _drift))
print("            (это баланс ЧИСЛА; он садится на единицу задолго до формы --")
print("             судить о стационарности по нему нельзя, см. блок ниже)")

# ---- СТАЦИОНАРНОСТЬ и ЧЕКПОЙНТ --------------------------------------------
#  Настоящий вердикт: <m> и alpha, сравненные между половинами хвоста.  Оба
#  привычных критерия -- M_out/M_in и dn_out/dev -- на шестидекадном холодном
#  прогоне отрапортовали успех на 0.2 промывки, когда <m> ещё росло по 14 за
#  промывку.  Подробности и таблица -- в AN.stationarity и в заметке [15].
ST4 = AN.stationarity(r4)
print(ST4["report"])

if SAVE_STATE:
    #  СОХРАНЯЕТСЯ ВСЕГДА, стационарен прогон или нет.  Отказывать в сохранении
    #  нестационарной прогоны было бы прямой ошибкой: итеративный порядок работы
    #  на ней и держится -- прогон, чекпойнт, проверка, ещё прогон с того же
    #  чекпойнта.  Если не сохранять, продолжать не с чего и приходится каждый
    #  раз начинать с дельты.
    #  Различие между «точкой продолжения» и «готовым состоянием» отмечается в
    #  meta и печатается, а не навязывается запретом.
    import os
    _p = AN.add_last_run(r4, os.path.basename(STATE_FILE).replace(".npz", ""),
                         drop_particles=False,
                         runs_dir=os.path.dirname(STATE_FILE) or "runs",
                         analysis=dict(checkpoint=True, flushes=ST4["flushes"],
                                       stationary=bool(ST4["ok"]),
                                       drift_mbar=ST4["mbar"][2],
                                       drift_alpha=ST4["alpha"][2]))
    print("\nЧЕКПОЙНТ СОХРАНЁН -> %s" % _p)
    if ST4["ok"]:
        print("  состояние СТАЦИОНАРНО: годится как старт измерительного прогона.")
        print("  дальше: START='state', SAVE_STATE=False -- и мерить.")
    else:
        print("  состояние НЕ стационарно: это ТОЧКА ПРОДОЛЖЕНИЯ, а не старт")
        print("  измерения.  Мерить с него можно, но это будет измерение")
        print("  переходного режима.")
        if np.isfinite(ST4.get("flushes_left", np.nan)):
            print("  дальше: START='state', SAVE_STATE=True -- ещё около %.1f промывки"
                  % ST4["flushes_left"])
            print("          = %.3g событий = %.1f ч при этой скорости"
                  % (ST4["flushes_left"] * _flush,
                     ST4["flushes_left"] * _flush / RATE4 / 3600.0))
        else:
            print("  дальше: START='state', SAVE_STATE=True -- ещё прогон;")
            print("          срок назвать нельзя, дрейф ещё не начал затухать.")
print("generations: снимков %g | overflow %.3g (%.3f%%)"
      % (r4["gen_snapshots"], r4["gen_overflow"],
         100 * r4["gen_overflow"] / max(r4["gen_num"].sum() + r4["gen_overflow"], 1)))

# ---- ЧЕСТНОСТЬ и ЗАКОН ОЖИДАНИЯ (только v5) --------------------------------
if WARM:
    _hn = np.asarray(r4["honest_num"]); _hm = np.asarray(r4["honest_mass"])
    print("честность : %.3f по числу, %.3f по массе на конец прогона"
          % (_hn[-1], _hm[-1]))
    print("            (это доля популяции с настоящей историей; поколения ниже"
          " считаются ТОЛЬКО по ней)")
    if _hn[-1] < 0.5:
        print("  ^^ больше половины -- ещё посев.  alpha и <dt> в порядке,"
              " а вот gen_* пока нет.")
    _we = np.asarray(r4["wait_events"])
    print("ожидание  : %.3g закрытых стоянок в %d бинах, медиана по бину %.3g"
          % (_we.sum(), int((_we > 0).sum()), np.median(_we[_we > 0])))
    if _we.sum() < 1e4:
        print("  ^^ мало: наклон <dt>(m) не определится.  Нужны десятки тысяч.")

F4  = steady_spectrum(r4)[0]
ir4 = BF.find_inertial_range(r4["centers"], F4)
gb4 = BF.guard_band(M_SINK4, M_INJ4, 0.4)
f4  = BF.fit_powerlaw(r4["centers"], F4, *gb4)
print("plateau [%9.3g,%9.3g] alpha = %+.3f +- %.3f на %.2f декадах | guard %+.3f  R2 = %.3f"
      % (ir4["m_lo"], ir4["m_hi"], ir4["alpha"], ir4["scatter"], ir4["decades"],
         f4["alpha"], f4["r2"]))
if WARM:
    print("            посеяно было %+.3f -- ушло на %+.3f; если не ушло, засев"
          " подсказал ответ" % (ALPHA_SEED, ir4["alpha"] - ALPHA_SEED))


In [ ]:
N_ISO_SHOW4 = 8      # isochrones on the spectrum panel

c4 = np.asarray(r4["centers"])
fig, a = plt.subplots(figsize=(6.0, 3.6))
idx, tauc, navail = pick_isochrones(r4, N_ISO_SHOW4)
print("isochrones: %d of %d age bins carry statistics -> drawing %d"
      % (navail, len(tauc), len(idx)))
draw_isochrones(a, r4, idx, tauc)
iso_sum = r4["iso_dndm"].sum(axis=0) / max(r4["iso_snapshots"], 1)
a.loglog(c4, np.where(iso_sum > 0, iso_sum * c4**2, np.nan), "-", lw=3.5, alpha=.35,
         color="0.4", zorder=2, label="sum of isochrones")
a.loglog(c4, np.where(F4 > 0, F4 * c4**2, np.nan), "o", ms=3, color="k", zorder=4,
         label="steady state")
if np.isfinite(ir4["m_lo"]):
    xs = np.logspace(np.log10(ir4["m_lo"]), np.log10(ir4["m_hi"]), 30)
    a.loglog(xs, anchor_amplitude(c4, F4, ir4["m_lo"], ir4["m_hi"], ir4["alpha"])
             * xs**(ir4["alpha"] + 2), "-", lw=2.2, color="C3",
             label=r"plateau $\alpha=%.2f$" % ir4["alpha"])
    a.loglog(xs, anchor_amplitude(c4, F4, ir4["m_lo"], ir4["m_hi"], PR["alpha"])
             * xs**(PR["alpha"] + 2), "--", lw=1.4, color="C1",
             label=r"theory $%.2f$" % PR["alpha"])
compensated_ylim(a, c4, F4)
a.set_xlabel("m"); a.set_ylabel(r"$m^2\,dN/dm$")
a.legend(fontsize=6, loc="lower left")
a.set_title("f = %.2g, w = %.2g, $N_0$ = %d" % (F_RATIO4, W_SPLIT, N04), fontsize=9)
fig.tight_layout()

---
## Закон ожидания: $b$ как время

$\langle\Delta t\rangle(m)$ — среднее время, которое частица проводит при массе $m$
между двумя событиями, меняющими её массу. Против степенного фона, по однородности
ядра,

$$\nu(m)=\int_{f m}^{m}K(m,m')\,n(m')\,dm'\;\sim\;m^{\lambda+\alpha+1},$$

и интеграл — чистое число **именно потому**, что порог $f>0$ делает его локальным.
Отсюда $\langle\Delta t\rangle=1/\nu\sim m^{1/b}$, а на неподвижной точке
$\alpha=-(3+\lambda)/2$ это даёт $1/b=(1-\lambda)/2$ — тот же $b$, что и замыкание,
как и должно быть.

От усиления $db/d\alpha=b^{2}$ эта дорога **не уходит**: $b=6$ — большое число,
сделанное из маленького, и это физика. Что она даёт — другое. Во-первых, это время,
измеренное прямо, а не показатель, вытянутый из плотности плюс аргумент о потоке; две
формулы, $1/b=2+\alpha$ и $1/b=-(1+\lambda+\alpha)$, совпадают только *на*
неподвижной точке, так что расхождение между ними измеряет, насколько прогон до неё не
дошёл. Во-вторых, величина **без памяти**: она не спрашивает, откуда частица взялась, и
потому честна на посеянной популяции с $t=0$ — ровно то, ради чего существует тёплый
старт. В-третьих, она **локальна**: сток портит только соседние с ним бины, а не всю
траекторию, и это то, что убило $\tau(g)$ в v4.

Платят за это динамическим диапазоном. $1/b=1/6$ значит, что $\langle\Delta t\rangle$
меняется всего в $10^{4/6}=4.6$ раза на четыре декады массы. Пологий наклон требует
длинного плеча, поэтому окно берётся **несимметричным**: декада отрезается у стока и
только $0{,}15$ у инжекции. Сток — жёсткое обрезание и достаёт вверх примерно на декаду;
инжекция — источник при одной массе и портит один бин.


In [ ]:
# ============================================================================
#  ЗАКОН ОЖИДАНИЯ.  b как ВРЕМЯ, а не через alpha.                    [v5]
# ============================================================================
#  <dt>(m) -- среднее время, которое частица сидит при массе m между двумя
#  событиями, МЕНЯЮЩИМИ её массу.  Против степенного фона скорость разрушения
#  тела массы m равна, по однородности ядра,
#
#      nu(m) = Int_{f m}^{m} K(m,m') n(m') dm'  ~  m^(lambda + alpha + 1),
#
#  интеграл -- чистое число ИМЕННО ПОТОМУ, что f > 0 делает его локальным.
#  Значит <dt> = 1/nu ~ m^(1/b), и на неподвижной точке 1/b = (1-lambda)/2.
#
#  ЧЕГО ЭТО НЕ ДАЁТ.  Наклон <dt> и alpha несут ОДИН И ТОТ ЖЕ показатель, и у
#  обоих db/dalpha = b^2.  От усиления не уходит никто: b = 6 -- большое число,
#  сделанное из маленького (1/6), и это физика, а не дефект оценки.  Поэтому
#  цитируется 1/b с ошибкой, а b -- производное.
#
#  ЧТО ЭТО ДАЁТ, и ради чего вообще написан v5:
#    * это ВРЕМЯ, измеренное прямо, а не показатель, вытащенный из плотности
#      плюс аргумент о потоке.  Две дороги -- 1/b = 2+alpha (замыкание) и
#      1/b = -(1+lambda+alpha) (кинетика) -- совпадают только НА неподвижной
#      точке, так что их сравнение ПРОВЕРЯЕТ замыкание, а не принимает его;
#    * оно БЕЗ ПАМЯТИ: не спрашивает, откуда частица взялась, поэтому годится
#      на посеянной популяции с t = 0.  tau(g) так не умеет, изохроны тоже;
#    * оно ЛОКАЛЬНО: сток цензурирует только соседние с ним бины, а не всю
#      траекторию -- ровно то, что убило tau(g) в v4 (b = 1.74 вместо 6).
#
#  ДВЕ ОЦЕНКИ одного и того же, и разница между ними -- сама по себе диагноз:
#    exposure  Int N_b dt / число законченных стоянок.  Не смещена цензурой.
#    interval  среднее по законченным интервалам.  Цензурирована на больших m,
#              что уполощает наклон и ЗАВЫШАЕТ b.  Если они разошлись -- прогон
#              короче самого долгого времени ожидания, надо длиннее.
if not WARM:
    print("ENGINE = 'cold': закона ожидания нет, эта ячейка ничего не делает.")
    WL = None
else:
    WL = AN.waiting_law(r4)                     # exposure, невзвешенный
    print(AN.waiting_report(r4, WL, alpha=ir4["alpha"], sigma_alpha=ir4["scatter"]))

    # ---- три дороги к b в одной таблице ------------------------------------
    _al, _sal = ir4["alpha"], ir4["scatter"]
    _rows = [("теория",                     0.5 * (1 - LAM),        np.nan),
             ("<dt>(m), измерено",          WL["p"],                WL["sigma_p"]),
             ("замыкание   2+alpha",        2 + _al,                _sal),
             ("кинетика  -(1+lam+alpha)",  -(1 + LAM + _al),        _sal)]
    print("\n%-26s %12s %10s %10s %10s" % ("дорога", "1/b", "sigma", "b", "sigma_b"))
    for _lab, _p, _sp in _rows:
        _b = 1.0 / _p if _p else np.nan
        print("%-26s %+12.5f %10.5f %10.3f %10.3f" % (_lab, _p, _sp, _b, _b * _b * _sp))
    print("\nsigma_b = b^2 sigma_(1/b).  Читать надо строку 1/b, а не строку b.")

    # ---- рисунок ------------------------------------------------------------
    _m, _dt, _n = WL["m"], WL["dt"], WL["n_events"]
    _ok = np.isfinite(_dt) & (_n > 0)
    fig, ax = plt.subplots(1, 3, figsize=(14, 3.4))

    ax[0].loglog(_m[_ok], _dt[_ok], "o", ms=3, color="0.5", label="все бины")
    ax[0].loglog(_m[WL["mask"]], _dt[WL["mask"]], "o", ms=4, color="C0", label="в фите")
    _xs = np.logspace(np.log10(WL["band"][0]), np.log10(WL["band"][1]), 30)
    ax[0].loglog(_xs, WL["A"] * _xs ** WL["p"], "-", lw=2, color="C3",
                 label=r"$1/b=%.4f\pm%.4f$" % (WL["p"], WL["sigma_p"]))
    _A0 = AN.anchor_amplitude(_m, _dt, WL["band"][0], WL["band"][1], 0.5 * (1 - LAM))
    ax[0].loglog(_xs, _A0 * _xs ** (0.5 * (1 - LAM)), "--", lw=1.3, color="C1",
                 label=r"теория $1/b=%.4f$" % (0.5 * (1 - LAM)))
    ax[0].axvspan(M_SINK4, WL["band"][0], color="0.85", zorder=0)
    ax[0].axvspan(WL["band"][1], M_INJ4, color="0.85", zorder=0)
    ax[0].set_xlabel("m"); ax[0].set_ylabel(r"$\langle\Delta t\rangle$")
    ax[0].legend(fontsize=6); ax[0].set_title("(a) закон ожидания", fontsize=9)

    #  Локальный наклон: единственная картинка, на которой видно, где закон есть,
    #  а где начинается сток.  Серое -- то, что выброшено окном.
    _l = np.log(_m); _ly = np.log(np.where(_dt > 0, _dt, np.nan))
    _sl = np.full_like(_l, np.nan)
    _sl[1:-1] = (_ly[2:] - _ly[:-2]) / (_l[2:] - _l[:-2])
    ax[1].semilogx(_m, _sl, "o-", ms=3, lw=.8, color="C0")
    ax[1].axhline(0.5 * (1 - LAM), ls="--", color="C1", lw=1.3,
                  label=r"теория $%.4f$" % (0.5 * (1 - LAM)))
    ax[1].axhline(WL["p"], ls="-", color="C3", lw=1.3, label="фит %.4f" % WL["p"])
    ax[1].axvspan(M_SINK4, WL["band"][0], color="0.85", zorder=0)
    ax[1].axvspan(WL["band"][1], M_INJ4, color="0.85", zorder=0)
    ax[1].set_ylim(-1.0, 1.0); ax[1].set_xlabel("m")
    ax[1].set_ylabel(r"$d\ln\langle\Delta t\rangle/d\ln m$")
    ax[1].legend(fontsize=6); ax[1].set_title("(b) локальный наклон = 1/b", fontsize=9)

    #  Честность во времени: сколько посева осталось.  На холодном прогоне это
    #  единица везде по построению, и рисовать нечего.
    _H = AN.honesty(r4)
    ax[2].plot(_H["n_out"] / max(N_SS4, 1), _H["num"], "o-", ms=3, lw=1,
               label="по числу")
    ax[2].plot(_H["n_out"] / max(N_SS4, 1), _H["mass"], "s-", ms=3, lw=1,
               label="по массе")
    ax[2].axhline(1.0, ls=":", color="0.4")
    ax[2].set_xlabel(r"$n_{\rm out}/N_{\rm ss}$"); ax[2].set_ylabel("честная доля")
    ax[2].set_ylim(-0.02, 1.05); ax[2].legend(fontsize=6)
    ax[2].set_title("(c) как уходит посев", fontsize=9)
    fig.tight_layout()


---
## Поколения

Номер разлома несёт **каждая** частица, один int32. Это та же величина, ради которой в
v2/v3 строились трейсеры, но снятая со всей популяции, а не с нескольких тысяч меченых
путей. Три ограничения трейсеров исчезают, а не решаются: число независимых деревьев
(его нельзя было поднять выше $n_{\rm out}/R$), полка у $m_{\rm inj}$, где сидело 92 %
времени пребывания, и цензурирование стоком — частица считается пока жива, и её
поколение не зависит ни от того, сколько она ждала, ни от того, доживёт ли она до конца.

**Взвешивание — единственное, что здесь надо не перепутать.** `gen_counts` считает оба
осколка, это взвешивание по **числу**, и один шаг равномерного дробления даёт
$\langle\ln\xi\rangle=-1$, ${\rm Var}=1$. `gen_mass` взвешивает по массе — ровно то, что
делал трейсер, следуя за куском с вероятностью $\xi$, — и даёт $-1/2$ и $1/4$. По массе
взято по умолчанию ещё и практически: пакет едет вдвое медленнее и остаётся выше стока
вдвое дольше по поколениям. На трёх декадах по числу линия ломается на $g=4$, по массе
держится до $g=10$.

Что теряется: гистограмма поколений — это **ансамбль** при фиксированном $g$, а не путь.
Корреляции между последовательными шагами одной траектории из неё не измерить. Для
мультипликативной картины это не потеря, потому что независимость шагов — это ровно то,
что делает $\langle x\rangle$ и ${\rm Var}(x)$ линейными по $g$, и эта линейность и есть
проверка.

Часы приходят бесплатно: $\langle\tau\rangle(g)$, время достижения поколения, копится в
момент события. Не путать со средним возрастом живущих сейчас в $g$ — тот насыщается на
времени жизни коробки и хранится отдельно как `gen_tau_live`.

In [ ]:
# ============================================================================
#  ПОКОЛЕНИЯ.  Всё ниже строится из gen_counts / gen_mass -- гистограммы,
#  которую несёт вся популяция, а не несколько тысяч меченых путей.
# ============================================================================
#  ВЗВЕШИВАНИЕ.  gen_counts считает ОБА осколка -- это взвешивание по ЧИСЛУ, и один
#  шаг равномерного дробления даёт <ln xi> = -1, Var = 1.  gen_mass взвешивает по
#  массе -- это ровно то, что делал трейсер, следуя за куском с вероятностью xi, -- и
#  даёт -1/2 и 1/4.  По массе взято по умолчанию не только из принципа: пакет едет
#  вдвое медленнее и остаётся выше стока вдвое дольше по поколениям.  Замерено на
#  трёх декадах: по числу линия ломается на g = 4, по массе держится до g = 10.
WEIGHT = "mass"        # <<-- 'mass' или 'number'
GEN_SHOW = 12          # сколько поколений рисовать на спектре

GEN = AN.generations(r4, weight=WEIGHT)
MOM = AN.gen_moments(GEN)
GRW = AN.gen_growth(GEN, MOM)
MU_TH, VAR_TH = AN.step_stats(W_SPLIT, WEIGHT)
c4 = GEN["centers"]

print("взвешивание %s: <ln xi> = %.4f, Var = %.4f  (точно, интегралом)" % (WEIGHT, MU_TH, VAR_TH))
print("%4s %11s %9s %9s %10s %10s %8s" %
      ("g", "вес", "<x>", "<x> теор", "Var", "Var теор", "reach"))
for g in GEN["g"][:GEN_SHOW + 6]:
    if not np.isfinite(MOM["mu"][g]):
        continue
    flag = "" if MOM["ok"][g] else "   <- пакет у стока"
    print("%4d %11.4g %9.3f %9.3f %10.3f %10.3f %8.3g%s"
          % (g, MOM["n"][g], MOM["mu"][g], MU_TH * g, MOM["var"][g], VAR_TH * g,
             MOM["reach_n"][g], flag))

# ---- (1) поколения на осях спектра -----------------------------------------
fig, ax = plt.subplots(1, 3, figsize=(14.5, 3.8))
gs = GEN["g"][MOM["ok"]][:GEN_SHOW] if MOM["ok"].any() else GEN["g"][1:GEN_SHOW + 1]
cm = plt.cm.viridis(np.linspace(0.04, 0.94, max(len(gs), 1)))
tot = GEN["H"].sum(axis=0)
_A = None
for col, g in zip(cm, gs):
    y = GEN["H"][g] / GEN["widths"] * c4        # ~ m * dN/dm при взвешивании по массе
    if _A is None:
        k0 = (c4 >= ir4["m_lo"]) & (c4 <= ir4["m_hi"]) & (tot > 0) & (F4 > 0)
        _A = np.median((F4[k0] * c4[k0]**2) / (tot[k0] / GEN["widths"][k0] * c4[k0]))
    ax[0].loglog(c4, np.where(y > 0, _A * y, np.nan), "-", lw=1.2, color=col,
                 label="g = %d" % g)
yt = _A * tot / GEN["widths"] * c4
ax[0].loglog(c4, np.where(yt > 0, yt, np.nan), "-", lw=3.5, alpha=.35, color="0.4",
             label="сумма по всем g")
ax[0].loglog(c4, np.where(F4 > 0, F4 * c4**2, np.nan), "o", ms=2.5, color="k",
             label="стационарный спектр")
compensated_ylim(ax[0], c4, F4)
ax[0].set_xlabel("m"); ax[0].set_ylabel(r"$m^2\,dN/dm$")
ax[0].legend(fontsize=5.5, ncol=2, loc="lower left")
ax[0].set_title("(a) поколения на спектре", fontsize=9)

for col, g in zip(cm, gs):
    h = GEN["H"][g]
    if h.sum() <= 0:
        continue
    ax[1].semilogy(np.log(c4 / M_INJ4), h / h.sum(), "-", lw=1.2, color=col)
    xx = np.linspace(MU_TH * g - 4 * np.sqrt(VAR_TH * g), MU_TH * g + 4 * np.sqrt(VAR_TH * g), 200)
    dx = np.log(c4[1] / c4[0])
    ax[1].semilogy(xx, dx * np.exp(-(xx - MU_TH * g)**2 / (2 * VAR_TH * g))
                   / np.sqrt(2 * np.pi * VAR_TH * g), "--", lw=.9, color=col, alpha=.75)
ax[1].axvline(np.log(M_SINK4 / M_INJ4), ls=":", color="0.3", lw=1.2)
ax[1].set_ylim(1e-5, 1); ax[1].set_xlabel(r"$x=\ln(m/m_{\rm inj})$")
ax[1].set_ylabel("доля поколения")
ax[1].set_title(r"(b) каждое поколение, пунктир $N(\mu g,\sigma^2 g)$", fontsize=9)

kk = MOM["ok"]
ax[2].plot(GEN["g"][kk], MOM["mu"][kk], "o", ms=4, color="C0", label=r"$\langle x\rangle$")
ax[2].plot(GEN["g"][kk], MOM["var"][kk], "s", ms=4, color="C1", label=r"${\rm Var}(x)$")
gg = GEN["g"][kk].astype(float)
ax[2].plot(gg, MU_TH * gg, "--", lw=1.2, color="C0", label=r"$%.3f\,g$" % MU_TH)
ax[2].plot(gg, VAR_TH * gg, "--", lw=1.2, color="C1", label=r"$%.3f\,g$" % VAR_TH)
ax[2].set_xlabel("g"); ax[2].legend(fontsize=6)
ax[2].set_title("(c) обе линейны по g, без подгонки", fontsize=9)
fig.tight_layout()

# ---- (2) часы и коллапс -----------------------------------------------------
fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
if np.isfinite(GRW["b"]):
    ax[0].plot(GRW["g"], GRW["tau"], "o", ms=5, label=r"$\langle\tau\rangle(g)$")
    _g = np.linspace(0, GRW["g"].max(), 200)
    ax[0].plot(_g, GRW["T"] * (1 - np.exp(-_g / GRW["scale"])), "-", lw=1.8, color="C3",
               label="фит b = %.2f (resid %.3f)" % (GRW["b"], GRW["resid"]))
    ax[0].plot(_g, GRW["T"] * (1 - np.exp(-_g / (PR["b"] / abs(MU_TH)))), "--", lw=1.3,
               color="C1", label="теория b = %.2f" % PR["b"])
ax[0].set_xlabel("g"); ax[0].set_ylabel(r"$\langle\tau\rangle$"); ax[0].legend(fontsize=6)
ax[0].set_title(r"(a) время достижения поколения", fontsize=9)

ax[1].semilogy(GEN["g"][1:], MOM["reach_n"][1:], "o-", ms=3, lw=1)
ax[1].semilogy(GEN["g"][1:], MOM["reach_n"][1] * 2.0**(GEN["g"][1:] - 1), "--", lw=1.2,
               color="C3", label=r"$2^g$ -- дробление удваивает")
ax[1].set_xlabel("g"); ax[1].set_ylabel("достигло g"); ax[1].legend(fontsize=6)
ax[1].set_title("(b) где сток начинает есть", fontsize=9)

for col, g in zip(cm, gs):
    h = GEN["H"][g]
    if h.sum() <= 0 or not np.isfinite(MOM["var"][g]) or MOM["var"][g] <= 0:
        continue
    z = (np.log(c4 / M_INJ4) - MOM["mu"][g]) / np.sqrt(MOM["var"][g])
    ax[2].semilogy(z, h / h.sum() / (np.log(c4[1] / c4[0]) / np.sqrt(MOM["var"][g])),
                   "-", lw=1.2, color=col)
zz = np.linspace(-4.5, 4.5, 300)
ax[2].semilogy(zz, np.exp(-zz**2 / 2) / np.sqrt(2 * np.pi), "k--", lw=1.4, label="N(0,1)")
ax[2].set_xlim(-4.5, 4.5); ax[2].set_ylim(1e-4, 1)
ax[2].set_xlabel(r"$z=(x-\mu g)/(\sigma\sqrt{g})$"); ax[2].legend(fontsize=6)
ax[2].set_title("(c) коллапс: одна кривая или нет", fontsize=9)
fig.tight_layout()

print()
if np.isfinite(GRW["b"]):
    print("b = %.2f из tau(g) на g = %d..%d (resid %.3f против прямой %.3f)"
          % (GRW["b"], int(GRW["g"].min()), int(GRW["g"].max()),
             GRW["resid"], GRW["resid_linear"]))
_beta = -(1.0 + ir4["alpha"]); _bc = 1.0/(1.0-_beta) if _beta < 1 else np.inf
print("b = %.1f +- %.1f из замыкания на alpha (db/dalpha = b^2)   теория %.2f"
      % (_bc, _bc**2 * ir4["scatter"], PR["b"]))
if WL:
    #  Ставится РЯДОМ намеренно.  tau(g) цензурируется длиной прогона, замыкание
    #  усилено в b^2 раз, <dt> ни то ни другое -- но зато у него своё окно.  Три
    #  числа, три разные болезни; сходятся -- значит ни одна не доминирует.
    print("b = %.2f +- %.2f (стат) +- %.2f (окно) из <dt>(m) ~ m^(1/b) на %.2f декадах"
          % (WL["b"], WL["sigma_b"], WL["sigma_b_sys"], WL["decades"]))
    print("    1/b: измерено %+.5f | замыкание %+.5f | кинетика %+.5f | теория %+.5f"
          % (WL["p"], 2 + ir4["alpha"], -(1 + LAM + ir4["alpha"]), 0.5 * (1 - LAM)))
_need = 2 * PR["b"] / abs(MU_TH)
print("масштаб tau(g) равен %.0f поколений; пригодных сейчас %d -- %s"
      % (_need / 2, int(MOM["ok_flux"].sum()),
         "хватает" if MOM["ok_flux"].sum() > _need else "МАЛО, b не определится"))

# ---- сохранить прогон ------------------------------------------------------
KTAG = KERNEL.__name__.replace("kernel_", "")
add_last_run(r4, "open_fragmentation_conservative_%s_f%.2f" % (KTAG, F_RATIO4),
             analysis=dict(
                 alpha_plateau=ir4["alpha"], alpha_scatter=ir4["scatter"],
                 plateau_decades=ir4["decades"],
                 plateau_m_lo=ir4["m_lo"], plateau_m_hi=ir4["m_hi"],
                 alpha_guard=f4["alpha"], guard_r2=f4["r2"],
                 guard_lo=gb4[0], guard_hi=gb4[1],
                 alpha_theory=PR["alpha"], b_theory=PR["b"],
                 b_gen=GRW["b"], b_gen_resid=GRW["resid"],
                 gen_weight=WEIGHT, gen_ok=int(MOM["ok"].sum()),
                 # [v5] закон ожидания и честность.  Оба уходят в meta, так что
                 # тетрадь анализа читает их, не пересчитывая прогон.
                 engine=BF.__name__, warm=bool(WARM),
                 alpha_seed=(ALPHA_SEED if WARM else None),
                 honest_num=(float(np.asarray(r4["honest_num"])[-1]) if WARM else 1.0),
                 honest_mass=(float(np.asarray(r4["honest_mass"])[-1]) if WARM else 1.0),
                 wait_inv_b=(WL["p"] if WL else np.nan),
                 wait_inv_b_sigma=(WL["sigma_p"] if WL else np.nan),
                 wait_inv_b_window=(WL["p_spread"] if WL else np.nan),
                 wait_b=(WL["b"] if WL else np.nan),
                 wait_decades=(WL["decades"] if WL else np.nan),
                 wait_band_lo=(WL["band"][0] if WL else np.nan),
                 wait_band_hi=(WL["band"][1] if WL else np.nan),
                 wait_inv_b_interval=(WL["p_other"] if WL else np.nan),
                 gen_ok_flux=int(MOM["ok_flux"].sum()), gen_max=GEN_MAX,
                 mu_step=MU_TH, var_step=VAR_TH,
                 frag_age_rule=FRAG_AGE_RULE, f_ratio=F_RATIO4,
                 frag_split_width=W_SPLIT,
                 stationarity_dnout_dev=float(_slope), live_drift_2nd_half=float(_drift),
                 age_step=AGE_STEP, seed=SEED4, lam=LAM, N0=N04, N_ss=N_SS4,
                 m_inj=M_INJ4, m_sink=M_SINK4, mbar=MBAR4,
                 q=r4["meta"]["injection_rate"], t_c=t_c4, t_turn=t_turn4,
                 tau_res=tau_res4))

RESULTS["open frag f=%.2g" % F_RATIO4] = dict(
    b=GRW["b"], b_th=PR["b"], alpha=ir4["alpha"], alpha_th=PR["alpha"],
    scatter=ir4["scatter"], decades=ir4["decades"], alpha_gb=f4["alpha"],
    b_wait=(WL["b"] if WL else np.nan),
    inv_b_wait=(WL["p"] if WL else np.nan),
    inv_b_wait_sig=(WL["sigma_p"] if WL else np.nan),
    b_closure=_bc, warm=bool(WARM))

---
## Summary

One row per run. `plateau` is the mean local slope over the longest flat stretch of
$\Gamma(m)$, with its scatter and its width in decades; `guard` is the a-priori band
fit kept as an independent cross-check, and the two are expected to agree. `b` comes from the
trajectories, fitted as $\langle\tau\rangle(k)=T\,(1-e^{-k/2b})$ against the number of
splits — not from the isochrones, and not from a scan through the $(\tau,m)$ cloud,
which is a multiplicative random walk and has no exponent that makes it linear.

Read `b` against the closure value printed beside it rather than against the theory
alone. $\beta=-(1+\alpha)$ and $b=1/(1-\beta)$ give $db/d\alpha=b^2$, a factor
thirty-six at $b=6$: one percent on $\alpha$ moves $b$ by a full unit. Expect $\alpha$
to converge cleanly and $b$ not to, and quote $b$ to two significant figures at most.

Read `dec` before believing `plateau`. The expected failure is the fragmentation run
at $f=0$, where the non-local disruption rule drives the drift from the sink scale
and pushes the index towards $-2$ whatever the kernel; its narrow plateau is the
notebook announcing that the number next to it is a diagnosis rather than a
measurement.

Every run above has been written into `runs/`. The analysis notebooks there reload
those files and rebuild each figure separately, so nothing below needs to be re-run
to change a plot.

In [ ]:
#  Одна строка на прогон.  ТРИ дороги к b рядом, потому что они болеют по-разному:
#    b_gen   из tau(g) -- цензурируется длиной прогона (в v4 дало 1.74 вместо 6);
#    b_clos  из замыкания на alpha -- усилено в b^2 = 36 раз;
#    b_wait  из <dt>(m) ~ m^(1/b) -- прямое измерение времени, но пологий наклон.
#  Сходятся -- значит ни одна болезнь не доминирует.  Расходятся -- смотреть, какая.
hdr = ("%-20s | %6s %6s %6s %6s | %8s %6s %5s | %9s" %
       ("case", "b_gen", "b_clos", "b_wait", "b_th", "plateau", "+-", "dec", "1/b wait"))
print(hdr); print("-" * len(hdr))
for name, d in RESULTS.items():
    print("%-20s | %6.2f %6.2f %6.2f %6.2f | %+8.3f %6.3f %5.2f | %+.4f+-%.4f" %
          (name, d["b"], d.get("b_closure", np.nan), d.get("b_wait", np.nan), d["b_th"],
           d["alpha"], d["scatter"], d["decades"],
           d.get("inv_b_wait", np.nan), d.get("inv_b_wait_sig", np.nan)))

print("\nplateau = find_inertial_range (самый длинный плоский участок локального наклона).")
print("dec     = ширина плато в декадах; ниже ~1 показатель -- не измерение.")
print("b_gen   = из <tau>(g) по поколениям.  Цензурируется длиной прогона: если")
print("          прогон короче одного tau_res, tau(g) упирается в t_end и b занижен.")
print("b_clos  = 1/(2+alpha).  db/dalpha = b^2 = 36 при b = 6, поэтому один процент")
print("          на alpha -- целая единица на b.  Ошибка тут почти всегда больше самого")
print("          интересного эффекта.")
print("b_wait  = из <dt>(m) ~ m^(1/b), прямое измерение времени [только тёплый движок].")
print("          Не цензурировано и не требует истории, но наклон пологий (1/6), так что")
print("          читать надо колонку 1/b с ошибкой, а не b.")
print("\nВсе прогоны записаны в runs/.  Тетради анализа перечитывают эти файлы и строят")
print("каждый рисунок отдельно, так что менять картинку можно, не перезапуская прогон.")
